# PROJET-GNN

**Authors:**
- Yennipena Jérémie COULIBALY
- Michel Junior DELANOUE

**Objective:** Predict the country of an airport based on its longitude, latitude, and population using a Graph Convolutional Network (GCN) model.

---
## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Exploration](#2-data-loading--exploration)
3. [Geographic Visualization](#3-geographic-visualization)
4. [Problem Definition](#4-problem-definition)
5. [Data Preparation for GNN](#5-data-preparation-for-gnn)
6. [Class Imbalance Analysis](#6-class-imbalance-analysis)
7. [Data Cleaning & Stratified Split](#7-data-cleaning--stratified-split)
8. [Model Definition & Training](#8-model-definition--training)
9. [Results & Visualization](#9-results--visualization)
---

## 1. Setup & Imports

We start by importing all the necessary libraries, grouped by purpose for clarity.

In [ ]:
# --- Standard library ---
from collections import Counter

# --- Scientific computing ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
import seaborn as sns
import plotly.graph_objects as go

# --- Graph manipulation ---
import networkx as nx

# --- PyTorch (deep learning framework) ---
import torch
import torch as th
import torch.nn.functional as F
from torch import nn

# --- PyTorch Geometric (graph neural networks) ---
from torch_geometric.data import Data
from torch_geometric.nn import VGAE, GCNConv
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import from_networkx, to_networkx

# --- Scikit-learn (utilities) ---
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

---
## 2. Data Loading & Exploration

We load the airport graph from a GraphML file. Each node represents an airport with attributes:
- `lon`: longitude
- `lat`: latitude
- `population`: population of the associated city
- `country`: country label (our prediction target)
- `city_name`: name of the city

In [ ]:
# Load the graph from the GraphML file
G = nx.read_graphml("./airportsAndCoordAndPop.graphml", node_type=int)

# Display general information about the graph
print(G)
print(f"Node attributes: {list(G.nodes(data=True))[:1]}")

**Observations:**
- The graph contains **3,363 airports** connected by **13,547 edges** (routes).
- Each node has 5 attributes: coordinates (lon, lat), population, country, and city name.

---
## 3. Geographic Visualization

### 3.1 Static 2D Graph Visualization

A quick spring-layout visualization using NetworkX to get a sense of the graph's structure.

In [ ]:
# Compute node positions using a spring layout
pos = nx.spring_layout(G, k=3.0, seed=42)

# Draw the graph with labels, green nodes, and red edges
nx.draw(G, pos=pos, with_labels=True, node_color='green', edge_color='red')

### 3.2 Interactive Geographic Map with Plotly

Since the graph is inherently geographic (airports), we plot each airport on an interactive world map using its actual coordinates. This is more informative than the spring layout above.

In [ ]:
# Extract coordinates from node attributes
lons = [data['lon'] for _, data in G.nodes(data=True)]
lats = [data['lat'] for _, data in G.nodes(data=True)]

# Prepare hover text (city name, country, and population)
textes = [
    f"{data['city_name']}, {data['country']}<br>Population: {data['population']}"
    for _, data in G.nodes(data=True)
]

# Create the scatter trace for nodes
node_trace = go.Scattergeo(
    lon=lons,
    lat=lats,
    text=textes,
    mode='markers',
    marker=dict(
        size=5,
        color='blue',
        opacity=0.7
    ),
    hoverinfo='text'
)

# Assemble the figure
fig = go.Figure(data=[node_trace])
fig.update_layout(
    title='Airports Map',
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        showcountries=True,
        countrycolor='rgb(204, 204, 204)'
    ),
    height=700
)
fig.show()

---
## 4. Problem Definition

**Goal:** Train a GCN to predict the **country** of a city based on:
- **Node features:** geographic coordinates (lon, lat) and population.
- **Graph structure:** relationships between cities (flight routes).

**Intuition:** Cities in the same country tend to be geographically close and densely connected to each other. The graph structure therefore carries valuable information that a purely feature-based classifier (e.g., logistic regression) would miss.

---
## 5. Data Preparation for GNN

We convert the NetworkX graph into a PyTorch Geometric `Data` object. The `from_networkx` utility automatically maps node attributes to tensors.

In [ ]:
# Convert NetworkX graph to PyG Data object
data = from_networkx(
    G,
    group_node_attrs=["lon", "lat", "population"],
)
print(data)

**Notes:**
- `edge_index` has shape `[2, 27094]` — the graph is treated as **undirected**, so PyG stores both directions, doubling the edge count (13,547 × 2 = 27,094).
- `x` has shape `[3363, 3]` — 3 features per node.
- `country` and `city_name` are kept as raw attributes (strings) for now.

---
## 6. Class Imbalance Analysis

Before training, we must understand the distribution of the target variable (`country`) to anticipate challenges.

In [ ]:
# Count the occurrences of each country
comptage = Counter(data.country)

# Display each country and its frequency
for pays, nb in comptage.most_common():
    print(f"{pays}: {nb}")

# Verify total
total = sum(comptage.values())
print(f"\nTotal: {total} rows (expected: {G.number_of_nodes()})")

In [ ]:
# Compute imbalance statistics
counts = np.array(list(comptage.values()))
print(f"Max: {counts.max()}, Min: {counts.min()}")
print(f"Imbalance ratio: {counts.max()/counts.min():.0f}:1")
print(f"Median: {np.median(counts)}, Mean: {counts.mean():.1f}")
print(f"Countries with <3 nodes: {(counts<3).sum()} / {len(counts)}")

**Key observations:**
- **Severe imbalance**: the USA alone has 650 airports, while 86 countries have fewer than 3.
- **212 distinct classes** in total, which is a *huge* number for a classification task.
- Many classes are so rare that a standard stratified split (70/15/15) would leave them out of some sets entirely.

### 6.1 Encoding Labels

We encode the country strings as integer indices.

In [ ]:
# Get labels in the node order of the graph
labels_str = [d['country'] for _, d in G.nodes(data=True)]

# Build mapping: country -> index
classes = sorted(set(labels_str))
pays2idx = {p: i for i, p in enumerate(classes)}
idx2pays = {i: p for p, i in pays2idx.items()}

# Encode labels as a tensor
y = torch.tensor([pays2idx[p] for p in labels_str], dtype=torch.long)
print(f"Number of classes: {len(classes)}")
print(f"Number of nodes: {len(y)}")

**Question raised:** *How should we handle countries with only one occurrence?*
- Keep them as-is?
- Merge them? By population? By region?

This will be addressed in the next section.

---
## 7. Data Cleaning & Stratified Split

Given the extreme imbalance, we take the following steps:

1. **Normalize features** — mean/std standardization is essential for GCN convergence.
2. **Encode labels** — convert strings to integer class IDs.
3. **Filter rare classes** — drop classes with fewer than `MIN_COUNT` examples.
4. **Reindex labels** — remap remaining class IDs to a dense range `[0, K-1]`.
5. **Filter edges** — keep only edges whose *both* endpoints are kept.
6. **Stratified split** — ensure *every* class appears in train/val/test (≥1 example each).

In [ ]:
# ============================================================
# 1. Normalize features (mean/std standardization)
# ============================================================
mean = data.x.mean(dim=0, keepdim=True)
std  = data.x.std(dim=0, keepdim=True) + 1e-9
data.x = (data.x - mean) / std

# ============================================================
# 2. Extract labels (country)
# ============================================================
nodes = list(G.nodes(data=True))
countries = [d['country'] for _, d in nodes]

le = LabelEncoder()
data.y = th.tensor(le.fit_transform(countries), dtype=th.long)

# ============================================================
# 3. Filter: keep only classes with >= MIN_COUNT examples
# ============================================================
MIN_COUNT = 5   # must be >= 3 to guarantee >=1 example per split

unique, counts = np.unique(data.y.numpy(), return_counts=True)
keep_classes = unique[counts >= MIN_COUNT]
print(f"Classes before filtering: {len(unique)}")
print(f"Classes kept:             {len(keep_classes)}")

# Mask on nodes to keep
node_mask = th.isin(data.y, th.tensor(keep_classes))
print(f"Nodes before: {data.num_nodes} | Nodes kept: {node_mask.sum().item()}")

# ============================================================
# 4. Dense reindexing of labels 0..K-1
# ============================================================
label_map = th.full((int(data.y.max().item()) + 1,), -1, dtype=th.long)
label_map[th.tensor(keep_classes)] = th.arange(len(keep_classes))
new_y = label_map[data.y[node_mask]]

# Filter features
new_x = data.x[node_mask]

# ============================================================
# 5. Filter edges: keep only those whose BOTH endpoints are kept
# ============================================================
old_to_new = th.full((data.num_nodes,), -1, dtype=th.long)
old_to_new[node_mask] = th.arange(node_mask.sum().item())

edge_mask = node_mask[data.edge_index[0]] & node_mask[data.edge_index[1]]
new_edge_index = old_to_new[data.edge_index[:, edge_mask]]

print(f"Edges before: {data.edge_index.shape[1]} | Edges kept: {new_edge_index.shape[1]}")

# ============================================================
# 6. Rebuild the Data object
# ============================================================
data = Data(x=new_x, edge_index=new_edge_index, y=new_y)
print(f"\n{data}")
print(f"Final classes: {data.y.max().item() + 1}")

### 7.1 Stratified Split with Minimum Coverage Guarantee

A naive `train_test_split` would fail for classes with very few examples. We implement a **custom stratified split** ensuring that even the rarest class appears in all three sets.

In [ ]:
# ============================================================
# 7. GUARANTEED STRATIFIED SPLIT: >=1 example per class in train/val/test
# ============================================================
num_nodes   = data.num_nodes
num_classes = int(data.y.max().item()) + 1
y_np        = data.y.numpy()

rng = np.random.default_rng(42)

train_idx, val_idx, test_idx = [], [], []
for c in range(num_classes):
    idx_c = np.where(y_np == c)[0]
    rng.shuffle(idx_c)
    n = len(idx_c)

    # Proportional 70/15/15 split with a minimum of 1 per subset
    n_val_c  = max(1, int(round(0.15 * n)))
    n_test_c = max(1, int(round(0.15 * n)))
    n_train_c = n - n_val_c - n_test_c

    # Safety check for very small classes (n < 3)
    if n_train_c < 1:
        n_train_c = 1
        remaining = n - 1
        n_val_c  = max(1, remaining // 2)
        n_test_c = remaining - n_val_c
        if n_test_c < 1:   # case n == 2
            n_test_c = 1
            n_val_c  = n - n_train_c - n_test_c

    train_idx.extend(idx_c[:n_train_c])
    val_idx.extend(idx_c[n_train_c:n_train_c + n_val_c])
    test_idx.extend(idx_c[n_train_c + n_val_c:])

# Build boolean masks
data.train_mask = th.zeros(num_nodes, dtype=th.bool)
data.val_mask   = th.zeros(num_nodes, dtype=th.bool)
data.test_mask  = th.zeros(num_nodes, dtype=th.bool)
data.train_mask[train_idx] = True
data.val_mask[val_idx]     = True
data.test_mask[test_idx]   = True

# ============================================================
# 8. VERIFICATION: every class must be present in every split
# ============================================================
print("\n--- Per-class check ---")
ok = True
for c in range(num_classes):
    tr = (data.y[data.train_mask] == c).sum().item()
    va = (data.y[data.val_mask]   == c).sum().item()
    te = (data.y[data.test_mask]  == c).sum().item()
    if tr < 1 or va < 1 or te < 1:
        ok = False
        print(f"Class {c}: train={tr} val={va} test={te}")
assert ok, "Some classes are missing from a split!"
print("Every class is present in train / val / test")

# ============================================================
# 9. Final summary
# ============================================================
print(f"\nNum nodes  : {data.num_nodes}")
print(f"Num edges  : {data.edge_index.shape[1]}")
print(f"Num classes: {num_classes}")
print(f"Train/Val/Test : {data.train_mask.sum().item()}/{data.val_mask.sum().item()}/{data.test_mask.sum().item()}")

# Distribution of remaining classes (overall)
unique, counts = np.unique(data.y.numpy(), return_counts=True)
print(f"Min/Max/Median examples per class: {counts.min()}/{counts.max()}/{int(np.median(counts))}")

# Distribution per split (useful for spotting rare classes)
for name, mask in [("train", data.train_mask),
                   ("val",   data.val_mask),
                   ("test",  data.test_mask)]:
    _, cnt = np.unique(data.y[mask].numpy(), return_counts=True)
    print(f"{name:5s} -> min/max per class: {cnt.min()}/{cnt.max()}")

**Justification for `MIN_COUNT = 5`:**
- With 3 examples per class, we could not have 1 train / 1 val / 1 test.
- With `MIN_COUNT = 5`, we get at minimum 3 train / 1 val / 1 test — enough for the model to see each class a few times during training.

---
## 8. Model Definition & Training

### 8.1 GCN Architecture

We use a **2-layer Graph Convolutional Network**:

```
Input (features) → GCNConv → ReLU → Dropout → GCNConv → LogSoftmax → Output
```

- **First GCN layer** aggregates information from 1-hop neighbors.
- **Second GCN layer** aggregates from 2-hop neighbors (i.e., neighbors of neighbors).
- **Dropout** prevents overfitting.
- **LogSoftmax** produces log-probabilities for the NLL loss.

In [ ]:
class Model(nn.Module):
    """
    2-layer Graph Convolutional Network for node classification.

    Architecture:
        x -> GCNConv -> ReLU -> Dropout -> GCNConv -> LogSoftmax
    """
    def __init__(self, in_channels, hidden_channels, out_channels=3):
        super().__init__()
        # First GCN layer: maps input features to hidden dimension
        self.conv1 = GCNConv(in_channels, hidden_channels)
        # Second GCN layer: maps hidden dimension to number of classes
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # Layer 1: convolution + activation
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        # Dropout for regularization (only active during training)
        x = F.dropout(x, p=0.5, training=self.training)
        # Layer 2: convolution -> log-probabilities
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

### 8.2 Training Loop

The training function:
- Uses **Adam optimizer** with L2 regularization (`weight_decay`).
- Computes **NLL loss** (negative log-likelihood) on the training set.
- Tracks **train and validation** loss + accuracy at each epoch.
- Correctly sets `model.eval()` during validation (disables dropout).

In [ ]:
def train(data, model, epochs):
    """
    Train the model and return training history.

    Args:
        data:   PyG Data object with train_mask / val_mask.
        model:  the GCN model.
        epochs: number of training iterations.

    Returns:
        model, train_losses, val_losses, train_accs, val_accs
    """
    optimizer = th.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    model.train()

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(epochs):
        # --- Training step ---
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # --- Validation step (no gradients, no dropout) ---
        model.eval()
        with th.no_grad():
            out_eval = model(data.x, data.edge_index)
            val_loss = F.nll_loss(out_eval[data.val_mask], data.y[data.val_mask])
        model.train()

        # --- Log metrics ---
        train_losses.append(loss.item())
        val_losses.append(val_loss.item())

        train_acc = (out[data.train_mask].argmax(1) == data.y[data.train_mask]).float().mean().item()
        val_acc   = (out_eval[data.val_mask].argmax(1) == data.y[data.val_mask]).float().mean().item()
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | "
                  f"Train loss: {loss.item():.4f}, acc: {train_acc:.3f} | "
                  f"Val loss: {val_loss.item():.4f}, acc: {val_acc:.3f}")

    return model, train_losses, val_losses, train_accs, val_accs

### 8.3 Test Function

The test function evaluates the model on all three splits (train / val / test). Called **once at the end** to report final performance.

In [ ]:
def test(data, model):
    """
    Evaluate the trained model on train, val, and test sets.
    Should only be called ONCE after training is complete.
    """
    model.eval()
    with th.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
    acc_train = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
    acc_val   = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean().item()
    acc_test  = (pred[data.test_mask]  == data.y[data.test_mask]).float().mean().item()
    print(f"Train acc: {acc_train:.4f} | Val acc: {acc_val:.4f} | Test acc: {acc_test:.4f}")
    return acc_test

---
## 9. Results & Visualization

### 9.1 Instantiate, Train, and Evaluate the Model

We dynamically determine `in_channels` (number of features) and `out_channels` (number of classes), then train for 1000 epochs.

In [ ]:
# Dynamically determine input/output dimensions
in_channels  = data.x.shape[1]
out_channels = int(data.y.max().item() + 1)

print(f"in_channels={in_channels} | out_channels={out_channels}")

# Instantiate the model
model = Model(in_channels=in_channels, hidden_channels=16, out_channels=out_channels)

# Train and capture training history
model, train_losses, val_losses, train_accs, val_accs = train(data, model, epochs=1000)

# Evaluate on the test set (only once, at the end)
pred = test(data, model)

### 9.2 Loss and Accuracy Curves

We plot the training curves to diagnose potential issues:
- **Loss curves** show whether the model is learning or overfitting.
- **Accuracy curves** show how well the model generalizes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss: train vs val ---
axes[0].plot(train_losses, label="Train", color="steelblue")
axes[0].plot(val_losses,   label="Val",   color="crimson")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss (train vs val)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# --- Accuracy: train vs val ---
axes[1].plot(train_accs, label="Train", color="seagreen")
axes[1].plot(val_accs,   label="Val",   color="orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy (train vs val)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

### 9.3 Interpretation of Results

**Final performance (99-class classification):**

| Split | Accuracy |
|-------|----------|
| Train | 56.93%   |
| Val   | 53.86%   |
| Test  | **55.45%** |

**Analysis:**

1. **Random baseline** for 99 balanced classes would be ~1%. Our model achieves **~55% accuracy**, which is far above chance — the GCN has clearly learned meaningful patterns.

2. **Train ≈ Val ≈ Test** — no significant overfitting or underfitting. The small gap (1–3%) indicates good generalization.

3. **Loss decreasing smoothly** — training is stable, no divergence or oscillation.

4. **Remaining ~45% error** likely comes from:
   - **Class imbalance**: the USA alone has 454 training examples, while 86 classes have <5. The model is biased toward frequent classes.
   - **Geographic overlap**: some countries are geographically adjacent (e.g., many European countries, US/Canada border areas).
   - **Feature limitations**: 3 features (lon, lat, population) may not suffice to disambiguate all 99 classes.

**Potential improvements (future work):**
- Use **class weights** in the loss to penalize misclassification of rare classes more heavily.
- Add **richer node features** (e.g., country encoding, time zone, language).
- Use **GraphSAGE** or **GAT** instead of GCN — attention mechanisms often handle heterogeneous neighborhoods better.
- Try **hierarchical classification** (continent → region → country) to reduce the branching factor.
- Compute **macro-F1** instead of accuracy to better reflect performance on rare classes.

---
## Summary

| Aspect | Value |
|--------|-------|
| Dataset | 3,363 airports, 13,547 undirected routes |
| Task | Node classification (predict country) |
| Features | (longitude, latitude, population) — standardized |
| Classes (after filtering) | 99 |
| Nodes retained | 3,160 |
| Edges retained | 23,344 |
| Split | 2,204 / 478 / 478 (train / val / test) |
| Model | 2-layer GCN, 16 hidden units |
| Optimizer | Adam (lr=0.01, weight_decay=5e-4) |
| Epochs | 1,000 |
| **Test accuracy** | **55.45%** |

The GCN successfully leverages the graph structure to predict airport countries from geographic features, achieving performance far above random baselines, despite severe class imbalance.